In [ ]:
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import drive


drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch')
LOCAL_ROOT = Path('/content/charades_raw_from_drive')

DRIVE_DOWNLOADS = DRIVE_ROOT / 'downloads'
DRIVE_MANIFESTS = DRIVE_ROOT / 'manifests'

LOCAL_DOWNLOADS = LOCAL_ROOT / 'downloads'
LOCAL_RAW = LOCAL_ROOT / 'raw'
LOCAL_ANN_DIR = LOCAL_RAW / 'annotations'
LOCAL_VIDEO_DIR = LOCAL_RAW / 'videos_480p'
LOCAL_MANIFESTS = LOCAL_ROOT / 'manifests'

for p in [LOCAL_DOWNLOADS, LOCAL_RAW, LOCAL_ANN_DIR, LOCAL_VIDEO_DIR, LOCAL_MANIFESTS]:
    p.mkdir(parents=True, exist_ok=True)

ANNOTATION_ZIP_SRC = DRIVE_DOWNLOADS / 'Charades.zip'
VIDEO_ZIP_SRC = DRIVE_DOWNLOADS / 'Charades_v1_480.zip'

ANNOTATION_ZIP_DST = LOCAL_DOWNLOADS / 'Charades.zip'
MANIFEST_FILES = [
    'charades_raw_manifest.json',
    'charades_raw_train_manifest.json',
    'charades_raw_val_manifest.json',
    'charades_raw_test_manifest.json',
    'charades_raw_survey.json',
    'charades_class_map.json',
    'charades_class_names.txt',
]

print('DRIVE_ROOT =', DRIVE_ROOT)
print('LOCAL_ROOT =', LOCAL_ROOT)
print('annotation zip on drive exists =', ANNOTATION_ZIP_SRC.exists())
print('video zip on drive exists =', VIDEO_ZIP_SRC.exists())
print('manifests on drive exists =', DRIVE_MANIFESTS.exists())


In [ ]:
def copy_file(src: Path, dst: Path):
    assert src.exists(), src
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print('skip existing:', dst.name)
        return
    shutil.copy2(src, dst)
    print('copied:', src.name, '->', dst)


def unzip_file(zip_path: Path, out_dir: Path):
    assert zip_path.exists(), zip_path
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    print('unzipped:', zip_path.name, '->', out_dir)


def rewrite_manifest_video_paths(manifest_path: Path):
    rows = json.loads(manifest_path.read_text())
    for row in rows:
        row['video_path'] = str((LOCAL_VIDEO_DIR / 'Charades_v1_480' / f"{row['video_id']}.mp4").resolve())
    manifest_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False))
    print('rewrote video_path in', manifest_path.name)


# Copy annotations zip and manifest artifacts from Drive to local runtime.
copy_file(ANNOTATION_ZIP_SRC, ANNOTATION_ZIP_DST)
for name in MANIFEST_FILES:
    copy_file(DRIVE_MANIFESTS / name, LOCAL_MANIFESTS / name)
for name in ['charades_raw_manifest.json', 'charades_raw_train_manifest.json', 'charades_raw_val_manifest.json', 'charades_raw_test_manifest.json']:
    rewrite_manifest_video_paths(LOCAL_MANIFESTS / name)


In [ ]:
# Unzip annotations locally.
unzip_file(ANNOTATION_ZIP_DST, LOCAL_ANN_DIR)

# Unzip videos directly from the zip stored on Drive into local runtime.
unzip_file(VIDEO_ZIP_SRC, LOCAL_VIDEO_DIR)

print('local annotations dir:', LOCAL_ANN_DIR)
print('local videos dir:', LOCAL_VIDEO_DIR)
print('local manifests dir:', LOCAL_MANIFESTS)


In [ ]:
# Quick sanity check from copied local artifacts.
manifest_path = LOCAL_MANIFESTS / 'charades_raw_survey.json'
train_manifest_path = LOCAL_MANIFESTS / 'charades_raw_train_manifest.json'
val_manifest_path = LOCAL_MANIFESTS / 'charades_raw_val_manifest.json'
class_map_path = LOCAL_MANIFESTS / 'charades_class_map.json'

assert manifest_path.exists(), manifest_path
assert train_manifest_path.exists(), train_manifest_path
assert val_manifest_path.exists(), val_manifest_path
assert class_map_path.exists(), class_map_path

survey = json.loads(manifest_path.read_text())
train_rows = json.loads(train_manifest_path.read_text())
val_rows = json.loads(val_manifest_path.read_text())
class_map = json.loads(class_map_path.read_text())

print(json.dumps({
    'num_train_videos_derived': len(train_rows),
    'num_val_videos_derived': len(val_rows),
    'num_classes': len(class_map['idx_to_name']),
    'avg_actions_per_video': survey['avg_actions_per_video'],
    'avg_video_length_sec': survey['avg_video_length_sec'],
}, indent=2))

print('sample train row:', {k: train_rows[0][k] for k in ['video_id', 'split', 'num_actions', 'length']})
print('sample video path exists:', Path((LOCAL_VIDEO_DIR / 'Charades_v1_480' / f"{train_rows[0]['video_id']}.mp4")).exists())
